# 🎙️ Open-Source ElevenLabs Studio (F5-TTS + Whisper ASR)
### Autonomous Zero-Shot Voice Cloning & High-Fidelity Text-to-Speech
---
> **Возможности:**
> * 🎯 **3–10s Zero-Shot Voice Cloning**: Точное копирование тембра и эмоций любого голоса.
> * 🇷🇺 🇬🇧 **Русский и Английский язык**: Естественные интонации, паузы и дыхание.
> * 🎙️ **Встроенный Whisper ASR**: Авто-распознавание сэмпла без ручного ввода.
> * 🎛️ **ElevenLabs Dark Studio UI**: Доступ с ПК и телефона по защищенной ссылке.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. 🚀 Проверка GPU и установка окружения
Нажмите кнопку **Play** слева для установки всех библиотек (занимает ~1 минуту).

In [ ]:
# @title ⚙️ Установка зависимостей (Run Setup)
!nvidia-smi

print('📦 Установка F5-TTS, Whisper ASR, Vocos и Gradio...')
!pip install -q git+https://github.com/SWivid/F5-TTS.git
!pip install -q openai-whisper gradio soundfile torchaudio accelerate vocos
!apt-get install -y -qq ffmpeg

print('✅ Все библиотеки успешно установлены!')


## 2. 🧠 Загрузка моделей в память GPU
Загружаем F5-TTS, нейровокодер Vocos и Whisper ASR.

In [ ]:
# @title 📥 Инициализация моделей
import os
import torch
import soundfile as sf
import whisper
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process
from huggingface_hub import hf_hub_download

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f'⚡ Устройство: {device} ({dev_name})')

print('📥 Загрузка Vocos...')
vocoder = load_vocoder(vocoder_name='vocos', device=device)

print('📥 Загрузка весов F5-TTS Base (Hugging Face / SWivid)...')
model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
ckpt_path = hf_hub_download(repo_id='SWivid/F5-TTS', filename='F5TTS_Base/model_1200000.safetensors')
model = load_model(DiT, model_cfg, ckpt_path=ckpt_path, device=device)

print('📥 Загрузка Whisper ASR (авто-распознавание сэмпла)...')
asr_model = whisper.load_model('base', device=device)

print('🔥 Все модели загружены и готовы к мгновенной генерации!')


## 3. 🎛️ Запуск ElevenLabs Web Studio (Gradio UI)
После запуска появится ссылка **`https://xxxx.gradio.live`** для работы в браузере.

In [ ]:
# @title 🎛️ Запуск веб-студии (Launch Gradio WebUI)
import gradio as gr
import numpy as np
import tempfile

def transcribe_audio(audio_path):
    if not audio_path:
        return ''
    try:
        res = asr_model.transcribe(audio_path)
        return res.get('text', '').strip()
    except Exception as e:
        return f'Ошибка транскрипции: {e}'

def synthesize_speech(ref_audio, ref_text, gen_text, speed, remove_silence):
    if not ref_audio:
        raise gr.Error('⚠️ Пожалуйста, загрузите или запишите образец голоса (3–10 секунд)!')
    if not gen_text or not gen_text.strip():
        raise gr.Error('⚠️ Введите текст для озвучки!')
    
    if not ref_text or not ref_text.strip():
        ref_text = transcribe_audio(ref_audio)
        if not ref_text:
            ref_text = 'Voice reference audio sample.'

    print(f'[TTS] Синтез речи: {gen_text[:50]}... | Скорость: {speed}x')
    
    final_wave, final_sample_rate, _ = infer_process(
        ref_audio=ref_audio,
        ref_text=ref_text,
        gen_text=gen_text,
        model_obj=model,
        vocoder=vocoder,
        speed=speed,
        device=device,
        remove_silence=remove_silence
    )
    
    out_path = tempfile.mktemp(suffix='.wav')
    sf.write(out_path, final_wave, final_sample_rate)
    return out_path, ref_text

custom_css = '''
.gradio-container { font-family: 'Inter', sans-serif; max-width: 1100px !important; margin: auto; }
#header-title { text-align: center; margin-bottom: 20px; }
#btn-generate { background: linear-gradient(135deg, #6366f1 0%, #4f46e5 100%); color: white; font-weight: bold; font-size: 16px; border-radius: 8px; }
#btn-transcribe { background: #374151; color: white; border-radius: 8px; }
'''

with gr.Blocks(title='ElevenLabs Open Studio | F5-TTS', css=custom_css, theme=gr.themes.Soft(primary_hue='indigo', neutral_hue='slate')) as demo:
    gr.Markdown(
        '''
        # 🎙️ ElevenLabs Open Studio (F5-TTS Zero-Shot Engine)
        ### Клонирование любого голоса по короткому 3–10 секундному сэмплу речи
        ''',
        elem_id='header-title'
    )
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### 1️⃣ Образец голоса (Voice Donor)')
            ref_audio = gr.Audio(
                label='Загрузить аудио или записать в микрофон (3-12 сек)',
                type='filepath',
                sources=['upload', 'microphone']
            )
            ref_text = gr.Textbox(
                label='Текст сэмпла (что говорится в аудио)',
                placeholder='Оставьте пустым для авто-распознавания через Whisper...',
                lines=2
            )
            btn_transcribe = gr.Button('🔍 Авто-распознать текст сэмпла (Whisper ASR)', elem_id='btn-transcribe')
            btn_transcribe.click(fn=transcribe_audio, inputs=[ref_audio], outputs=[ref_text])
            
            gr.Markdown('### 🎛️ Настройки генерации')
            speed_slider = gr.Slider(label='Скорость речи (Speed)', minimum=0.6, maximum=1.6, value=1.0, step=0.05)
            silence_cb = gr.Checkbox(label='Удалять лишнюю тишину (Remove Silence)', value=True)
            
        with gr.Column(scale=1):
            gr.Markdown('### 2️⃣ Текст для озвучки (Target Speech)')
            gen_text = gr.Textbox(
                label='Введите текст для генерации клонированным голосом (Русский / English / др.)',
                placeholder='Привет! Это полностью мой собственный аналог ElevenLabs на нейросети F5-TTS. Голос клонируется мгновенно и звучит максимально натурально!',
                lines=7
            )
            
            btn_generate = gr.Button('⚡ Сгенерировать речь клонированным голосом', elem_id='btn-generate', size='lg')
            
            gr.Markdown('### 3️⃣ Результат (Generated Audio)')
            out_audio = gr.Audio(label='Готовая озвучка (WAV / 24kHz)', type='filepath')
            
            btn_generate.click(
                fn=synthesize_speech,
                inputs=[ref_audio, ref_text, gen_text, speed_slider, silence_cb],
                outputs=[out_audio, ref_text]
            )

    gr.Markdown(
        '''
        ---
        💡 **Советы для идеального клонирования:**
        1. Используйте сэмпл голоса **без музыки, эха и посторонних шумов** длительностью от 4 до 10 секунд.
        2. Чем выразительнее и чище образец, тем точнее передаются эмоции и интонации.
        3. Если оставить текстовое поле сэмпла пустым, встроенный **Whisper** автоматически транскрибирует слова.
        '''
    )

demo.queue().launch(share=True, debug=False)


## 4. 🌐 REST API (Использование из Telegram-бота или скриптов)

In [ ]:
# @title 🌐 Пример вызова через Python API / Telegram-бот
# После запуска блока 3 скопируйте публичную ссылку xxxx.gradio.live
'''
from gradio_client import Client, handle_file

client = Client('https://YOUR_LINK.gradio.live')
result = client.predict(
    ref_audio=handle_file('my_sample.wav'),
    ref_text='',
    gen_text='Привет! Голос клонирован успешно.',
    speed=1.0,
    remove_silence=True,
    api_name='/synthesize_speech'
)
print('Аудио сохранено в:', result[0])
'''
